# Spam Mail Prediction AI

Trained from `mail_data.csv` using the same core workflow as the supplied notebook: TF-IDF feature extraction + Logistic Regression.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [ ]:
raw_mail_data = pd.read_csv('mail_data.csv')
mail_data = raw_mail_data.where(pd.notnull(raw_mail_data), '')
mail_data.head()

In [ ]:
mail_data['Category'] = mail_data['Category'].map({'spam': 0, 'ham': 1}).astype(int)
X = mail_data['Message'].astype(str)
Y = mail_data['Category']
print('Dataset shape:', mail_data.shape)
print('Spam:', (Y==0).sum(), '| Ham:', (Y==1).sum())

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=3, stratify=Y
)
print(X_train.shape, X_test.shape)

In [ ]:
feature_extraction = TfidfVectorizer(min_df=1, stop_words='english', lowercase=True)
X_train_features = feature_extraction.fit_transform(X_train)
X_test_features = feature_extraction.transform(X_test)
print('Train TF-IDF shape:', X_train_features.shape)

## Train Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_features, Y_train)

In [ ]:
train_pred = model.predict(X_train_features)
test_pred = model.predict(X_test_features)
print('Training accuracy:', accuracy_score(Y_train, train_pred))
print('Test accuracy:', accuracy_score(Y_test, test_pred))
print('Spam precision:', precision_score(Y_test, test_pred, pos_label=0))
print('Spam recall:', recall_score(Y_test, test_pred, pos_label=0))
print('Spam F1:', f1_score(Y_test, test_pred, pos_label=0))
print('\nConfusion matrix [rows=true, cols=predicted]:\n', confusion_matrix(Y_test, test_pred))
print('\n', classification_report(Y_test, test_pred, target_names=['spam','ham']))

## Predict a New Email

In [ ]:
input_mail = ['Free entry in a weekly competition! Text WIN to 87121 now.']
input_data_features = feature_extraction.transform(input_mail)
prediction = model.predict(input_data_features)[0]
prob = model.predict_proba(input_data_features)[0]
print('Prediction:', 'Spam mail' if prediction == 0 else 'Ham mail')
print('Spam probability:', prob[0])
print('Ham probability:', prob[1])